In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Jupyter内嵌绘图
%matplotlib inline

# 学术绘图全局配置
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 12,
    'axes.linewidth': 1.2,
    'grid.linewidth': 0.8,
    'grid.color': '#d3d3d3',
    'figure.dpi': 120,
    'savefig.dpi': 300,
    'savefig.bbox_inches': 'tight',
    'axes.axisbelow': True
})

# -------------------------- 核心配置 --------------------------
cases = ['F0', 'SA30', 'SD30']  # 三个工况
wind_angles = [0, 15, 30, 45, 60, 90]  # 六个风向角
# -------------------------------------------------------------

results = {case: [] for case in cases}

def calc_plane_average(csv_path):
    """读取单个CSV，计算平面平均风速（适配你的风洞CSV结构）"""
    # 读取CSV，前2行是说明，不做表头
    df = pd.read_csv(csv_path, header=None)
    # 取第3行（平均风速行）的16个通道数据
    avg_row = df.iloc[2, 1:17]
    # 计算算术平均
    return np.mean(avg_row)

# 批量读取所有18个文件
print("正在读取文件并计算...")
for case in cases:
    print(f"\n处理工况: {case}")
    for angle in wind_angles:
        filename = f"{case}_N_{angle}.csv"
        if not Path(filename).exists():
            print(f"  缺失文件: {filename}")
            results[case].append(np.nan)
            continue
        v_avg = calc_plane_average(filename)
        results[case].append(v_avg)
        print(f"  风向角 {angle}° → 平均风速: {v_avg:.4f} m/s")

# 归一化（以F0_0°为基准）
v0_baseline = results['F0'][0]
df_norm = pd.DataFrame(results, index=wind_angles) / v0_baseline
print(f"\n归一化基准值(F0_0°): {v0_baseline:.4f} m/s")

# 绘图
fig, ax = plt.subplots(figsize=(10, 6))
# 颜色和目标图完全匹配
colors = {'F0':'#ffc107', 'SA30':'#444444', 'SD30':'#000080'}

for case in cases:
    ax.plot(
        wind_angles, df_norm[case],
        color=colors[case], marker='o',
        linewidth=1.5, markersize=8, label=case
    )

# 图表美化
ax.set_title('Plane average 60mm N (λp=0.40)', fontsize=14, pad=15)
ax.set_xlabel('Wind angle (°)', fontsize=12)
ax.set_ylabel('Vave / Vave_F0(0.25_0°)', fontsize=12)
ax.set_xlim(0, 90)
ax.set_xticks(wind_angles)
ax.set_ylim(0, 2.5)
ax.set_yticks(np.arange(0, 2.6, 0.5))
ax.grid(axis='y')
ax.legend(loc='upper right', fontsize=11)

plt.show()

# 导出高清图
fig.savefig('wind_plot_3cases.png')
fig.savefig('wind_plot_3cases.pdf', format='pdf')
print("\n✅ 绘图完成，高清图已导出！")